# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## 1. Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
scale = 4
init_dask_cluster_eopf(scale=scale)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/a5556e52ebab4c4fb1dac92d5718975c/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [3]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

15:54:10.732 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/logging_config.yaml'.

15:54:10.735 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.short.yaml'.

15:54:10.736 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_configuration.yaml'.

15:54:10.737 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s1/iw_joborder.yaml'.

15:54:10.738 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/l0_processor_configuration.yaml'.

15:54:10.739 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'.

15:54:10.769 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0/config' to the bucket 'prefect-share' path 'sub/dir/users/jgaucher/l0/config/s3/s3_dordop_payload.yaml'

In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [11]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/jgaucher/code'


In [12]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor.yaml"

15:56:46.134 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette ~= 0.13.0" but found: "starlette 0.45.3"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment 'first-l0-processor/sprint21-first-l0-processor' successfully     │
│ created with id 'b7d40f4b-d5d7-4281-981c-309edc5a1f7d'.                      │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/b7d40f4b-d5d7-4281-981c-309edc5a1f7d


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor/sprint21-first-l0-processor'



In [7]:
deploy_name = "first-l0-processor/sprint21-first-l0-processor"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor/sprint21-first-l0-processor'


## 3. Run Prefect flow for S1 short data (~1 minute)

In [13]:
output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

Remove existing zarr products from: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'


In [14]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor/sprint21-first-l0-processor'...
Created flow run 'positive-deer'.
└── UUID: 8fc4e069-88ad-4d4e-be7a-4883c24a8875
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-03-07 15:56:51 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/8fc4e069-88ad-4d4e-be7a-4883c24a8875
Watching flow run 'positive-deer'...


15:56:51.289 | INFO    | prefect - Flow run is in state 'Scheduled'
15:56:56.305 | INFO    | prefect - Flow run is in state 'Scheduled'
15:57:01.323 | INFO    | prefect - Flow run is in state 'Pending'
15:57:06.336 | INFO    | prefect - Flow run is in state 'Running'
15:57:11.352 | INFO    | prefect - Flow run is in state 'Running'
15:57:16.370 | INFO    | prefect - Flow run is in state 'Running'
15:57:21.385 | INFO    | prefect - Flow run is in state 'Running'
15:57:26.400 | INFO    | prefect - Flow run is in state 'Running'
15:57:31.416 | INFO    | prefect - Flow run is in state 'Running'
15:57:36.431 | INFO    | prefect - Flow run is in state 'Running'
15:57:41.444 | INFO    | prefect - Flow run is in state 'Running'
15:57:46.458 | INFO    | prefect - Flow run is in state 'Running'
15:57:51.474 | INFO    | prefect - Flow run is in state 'Running'
15:57:56.491 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [10]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

Output products generated on: 's3://prefect-share/sub/dir/users/jgaucher/l0/output/s1.short'
Download reports locally: './l0/reports/s1.short'


## 4. Run Prefect flow for S1 full data (~30 minutes)

In [ ]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")
    
    local_report_dir = osp.join("./l0", "reports", "s1")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [ ]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")

    local_report_dir = osp.join("./l0", "reports", "s3")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway.scale_cluster(dask_cluster.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [5]:
from importlib import reload
debug_flow = True

In [14]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=4)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

Shutting down cluster 'ed7fe039898d4bf494d81a29e94e5c06' ...
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/37d57281c85b454ca770b3574fdb36fc/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [20]:
if debug_flow:
    import first_l0_processor
    reload(first_l0_processor)
    results = first_l0_processor.first_l0_processor(**s1_short)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.3  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


15:38:41.238 | INFO    | prefect.engine - Created flow run 'simple-alpaca' for flow 'first-l0-processor'

15:38:41.239 | INFO    | prefect.engine - View at http://prefect-server:4200/runs/flow-run/967eb194-30a2-49ec-a197-e8386e9d5229

15:38:41.309 | INFO    | Task run 'dummy_cadip_search-911' - Start cadip search

15:38:41.312 | INFO    | Task run 'dummy_auxip_search-33c' - Start auxip search

...

15:40:16.173 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:40:16.174 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:40:16.174 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:40:16.175 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client

15:40:16.175 | ERROR   | distributed.client - Failed to reconnect to scheduler after 30.00 seconds, closing client